# 44 · Asset-backed transformation tests

**Scenario:** Test schemas, NULL behavior, output invariants and unordered DataFrame equality using the real asset.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Test a reusable transformation

All test records are slices/derivatives of the actual Databricks asset. No miniature hand-entered test dataset replaces it.


In [ ]:
def transform(frame):
    return frame.filter('distance > 0 AND fare >= 0').select('trip_id','pickup_zip','fare','distance').withColumn('ratio',F.col('fare')/F.col('distance'))
out=transform(df)
assert out.columns==['trip_id','pickup_zip','fare','distance','ratio']
assert out.filter('distance <= 0 OR fare < 0 OR ratio IS NULL').count()==0
assert out.count()<=df.count()
print('Schema and value invariants passed')


**Expected check:** The transformation keeps its documented schema and validity contract.


## Step 3 — Test determinism and replay properties

Compare both directions with exceptAll; ordering should not matter for a relational transformation.


In [ ]:
rerun=transform(df)
assert out.exceptAll(rerun).limit(1).count()==0
assert rerun.exceptAll(out).limit(1).count()==0
double=transform(df.unionByName(df))
assert double.count()==2*out.count()
assert double.dropDuplicates(['trip_id']).count()==out.dropDuplicates(['trip_id']).count()
print('Determinism and asset replay checks passed')


**Expected check:** Replaying input doubles multiplicity but does not add new content fingerprints.


## Step 4 — Optionally run the included pytest suite

Add pytest to the notebook environment and set TEST_FILE to the absolute Workspace path of tests/test_asset_transformations.py in this Git folder. Fixtures below pass the actual asset DataFrame and the transformation defined above; no Spark mocks or replacement records are used.


In [ ]:
RUN_PYTEST=False
TEST_FILE=''
if RUN_PYTEST:
    assert TEST_FILE.startswith('/Workspace/') and TEST_FILE.endswith('/tests/test_asset_transformations.py')
    import pytest
    class AssetFixtures:
        @pytest.fixture(scope='session')
        def taxi_asset(self):
            return df
        @pytest.fixture(scope='session')
        def trip_transform(self):
            return transform
    status=pytest.main([TEST_FILE,'-q','-p','no:cacheprovider'],plugins=[AssetFixtures()])
    assert status==0, 'Asset-backed pytest suite failed'
else:
    print('Inline assertions passed; optional pytest runner skipped.')


**Expected check:** Enabled mode runs the checked-in pytest tests against the real asset on attached Databricks compute.


## Practice

Convert these checks into pytest tests with a session-scoped Databricks Spark fixture and a source-asset fixture.

## Solution guide

Use Databricks Connect or a Databricks job test task with the real source asset. Keep pure configuration tests local; do not mock Spark execution and call it an integration test.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
